# Particle-core model

In [ ]:
import copy

import numpy as np
import matplotlib.pyplot as plt
import scipy.integrate

plt.style.use("../style.mplstyle")

In [ ]:
class Lattice:
    def __init__(self, phase_advance: float, length: float = 1.0) -> None:
        self.length = length
        self.phase_advance = phase_advance
        self.wavenumber = self.phase_advance / self.length
        self.kappa = self.wavenumber ** 2

In [ ]:
class Envelope:
    def __init__(self, r: float, rp: float, emittance: float, perveance: float) -> None:
        self.r = r
        self.rp = rp
        self.emittance = emittance
        self.perveance = perveance

    def set_state_vec(self, v: np.ndarray) -> None:
        self.r = v[0]
        self.rp = v[1]

    def get_state_vec(self) -> np.ndarray:
        return np.array([self.r, self.rp])

In [ ]:
class Particles:
    def __init__(self, coords: np.ndarray) -> None:
        self.coords = coords

    def set_state_vec(self, v: np.ndarray) -> None:
        self.coords = np.reshape(v, self.coords.shape)

    def get_state_vec(self) -> np.ndarray:
        return np.ravel(self.coords)

In [ ]:
class Tracker:
    def __init__(self, lattice: Lattice) -> None:
        self.lattice = lattice
        self.envelope = None
        self.particles = None
        
    def set_state_vec(self, v: np.ndarray) -> None:
        self.envelope.set_state_vec(v[:2])
        self.particles.set_state_vec(v[2:])     

    def get_state_vec(self) -> None:
        return np.hstack([self.envelope.get_state_vec(), self.particles.get_state_vec()])

    def derivatives(self, s: float, v: np.ndarray) -> np.ndarray:        
        self.set_state_vec(v)
        vp = np.zeros_like(v)

        # Envelope
        vp[0] = self.envelope.rp
        vp[1] = -self.lattice.kappa * self.envelope.r + (self.envelope.emittance**2 / self.envelope.r**3) + self.envelope.perveance / self.envelope.r

        # Particles
        x, xp, y, yp = self.particles.coords.T
        
        r = np.sqrt(x**2 + y**2)
        inside = r <= self.envelope.r
        outside = ~inside
        
        kappa = self.lattice.kappa * np.ones_like(r)

        n_inside = np.count_nonzero(inside)
        if n_inside > 0:
            kappa[inside] -= self.envelope.perveance / self.envelope.r**2
        if n_inside < inside.shape[0]:
            kappa[outside] -= self.envelope.perveance / r[outside]**2

        vp[2::4] = xp
        vp[3::4]= -kappa * x
        vp[4::4] = yp
        vp[5::4]= -kappa * y
        return vp      

    def track(self, envelope: Envelope, particles: Particles, periods: int = 1, steps: int = 100):
        self.envelope = envelope
        self.particles = particles
        
        positions = np.linspace(0.0, periods * self.lattice.length, periods * steps + 1)
        
        result = scipy.integrate.solve_ivp(
            fun=tracker.derivatives,
            t_span=(positions[0], positions[-1]),
            t_eval=positions,
            y0=self.get_state_vec(),
        )
        self.set_state_vec(result.y[:, -1])
        
        history = {}
        history["s"] = result.t
        history["envelope_r"] = result.y[0, :]
        history["envelope_rp"] = result.y[1, :]
        history["particles"] = result.y[2:, :].reshape(
            periods * steps + 1,
            self.particles.coords.shape[0], 
            self.particles.coords.shape[1],
        )
        return history

    def track_tbt(self, envelope: Envelope, particles: Particles, periods: int = 1, length: float = None) -> None:
        self.envelope = envelope
        self.particles = particles

        if length is None:
            length = self.lattice.length

        history = {}
        history["envelope_r"] = []
        history["envelope_rp"] = []
        history["particles"] = []
        
        for period in range(periods + 1):
            if period > 0:
                result = scipy.integrate.solve_ivp(
                    fun=tracker.derivatives,
                    t_span=(0.0, length),
                    y0=self.get_state_vec(),
                )
                self.set_state_vec(result.y[:, -1])

            history["envelope_r"].append(self.envelope.r)
            history["envelope_rp"].append(self.envelope.rp)
            history["particles"].append(self.particles.coords.copy())

        for key in history:
            history[key] = np.array(history[key])
            
        return history

In [ ]:
def get_equilibrium_radius(emittance: float, k: float) -> float:
    return np.sqrt(emittance / k)


def get_perveance(k: float, k0: float, radius: float) -> None:
    return (k0**2 - k**2) * radius**2

In [ ]:
length * 

In [ ]:
length = 1.0

sigma0 = np.radians(80.0)
sigma = sigma0 * 0.3

k0 = sigma0 / length
k = sigma / length
k_env = np.sqrt(2.0 * (k**2 + k0**2))

wavelength = 1.0 / k
print(wavelength)

emittance = 1e-6
eq_radius = get_equilibrium_radius(emittance, k)
perveance = get_perveance(k, k0, eq_radius)

lattice = Lattice(phase_advance=sigma0, length=length)
radius = 0.62 * eq_radius
envelope = Envelope(r=radius, rp=0.0, emittance=emittance, perveance=perveance)
particles = Particles(np.zeros((10, 4)))

tracker = Tracker(lattice=lattice)
history = tracker.track(envelope=envelope, particles=particles, periods=10)

fig, ax = plt.subplots(figsize=(4, 3))
ax.plot(history["s"], history["envelope_r"] * 1000.0)
ax.set_xlabel("Distance [m]")
ax.set_ylabel("Beam radius [mm]")
ax.set_ylim(0.0, 2.0 * 1000.0 * np.mean(history["envelope_r"]))
plt.show()

In [ ]:
length = 1.0

sigma0 = np.radians(80.0)

sigma = sigma0 * np.linspace(0.3, 0.9, 100)

k0 = sigma0 / length
k = sigma / length

k_env = np.sqrt(2.0 * (k**2 + k0**2))

fig, ax = plt.subplots(figsize=(4, 3))
ax.fill_between(sigma / sigma0, k_env / k, k_env / k0)
ax.set_ylim(0.0, 4.0)
plt.show()

In [ ]:
length = 1.0

sigma0 = np.radians(80.0)
sigma = sigma0 * 0.3

k0 = sigma0 / length
k = sigma / length
k_env = 2.0 * (k**2 + k0**2)

wavelength = 1.0 / k
print(wavelength)

emittance = 1e-6
eq_radius = get_equilibrium_radius(emittance, k)
perveance = get_perveance(k, k0, eq_radius)

lattice = Lattice(phase_advance=sigma0, length=length)

radius = 0.62 * eq_radius

envelope = Envelope(r=radius, rp=0.0, emittance=emittance, perveance=perveance)

n = 25
particles = np.zeros((n, 4))
particles[:, 0] = np.linspace(0.0, 4.0 * eq_radius, n)

particles = Particles(particles)

tracker = Tracker(lattice=lattice)
history = tracker.track_tbt(envelope=envelope, particles=particles, length=wavelength, periods=100)

fig, ax = plt.subplots(figsize=(4, 4))
for index in range(history["particles"].shape[1]):
    ax.scatter(
        1000.0 * history["particles"][:, index, 0],
        1000.0 * history["particles"][:, index, 1],
        s=3,
        c="black",
        ec="none"
    )

xmax = 1.25 * 1000.0 * np.max(history["particles"], axis=(0, 1))
limits = list(zip(-xmax, xmax))
ax.set_xlim(limits[0])
ax.set_ylim(limits[1])
plt.show()

fig, ax = plt.subplots(figsize=(4, 3))
ax.plot(history["envelope_r"][:10] * 1000.0, marker=".")

ax.set_xlabel("Period")
ax.set_ylabel("Beam radius [mm]")
ax.set_ylim(0.0, 2.0 * 1000.0 * np.mean(history["envelope_r"]))
plt.show()